# Lecture 06: every figure in the slides

These cells are the code that produced the nonlinear regression figures in the deck, with each file write replaced by an inline plot. The notebook runs independently of the slide repository.

In [ ]:
# Colab opens this notebook on its own, so fetch the data files that sit
# beside it in the course repository. Running locally, this does nothing.
from pathlib import Path as _Path
import urllib.request as _req
_RAW = 'https://raw.githubusercontent.com/ZhuoranYang/sds265-fall26/main/demos/06-nonlinear-regression/'
for _name in ['Auto.csv', 'Wage.csv']:
    if not _Path(_name).exists():
        _req.urlretrieve(_RAW + _name, _name)

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The course palette and figure sizes, copied from slides/figures/sds265_style.py
BLUE, BLUE_BRIGHT, RED, GREEN = '#00356B', '#286DC0', '#7A3E47', '#5B7864'
INK, MUTED, RULE, BLUE_PALE = '#202A35', '#66727E', '#D8E0E8', '#F2F6FA'
BLUE_FILL, RED_FILL, GREEN_FILL = '#8C9FB1', '#B0846F', '#9AAE96'
MORANDI_BLUE, MORANDI_CLAY, MORANDI_SAGE = '#8C9FB1', '#B0846F', '#9AAE96'
MORANDI_MAUVE, MORANDI_SAND, MORANDI_STONE = '#A98FA0', '#C7B392', '#8A8F8A'
MORANDI = [MORANDI_BLUE, MORANDI_CLAY, MORANDI_SAGE, MORANDI_MAUVE, MORANDI_SAND, MORANDI_STONE]
MORANDI_EDGE = {MORANDI_BLUE: '#5C7186', MORANDI_CLAY: '#835747', MORANDI_SAGE: '#6B8067',
                MORANDI_MAUVE: '#7C6373', MORANDI_SAND: '#9A8763', MORANDI_STONE: '#5F645F'}
CYCLE = MORANDI
TEXTWIDTH, TEXTHEIGHT = 5.63, 3.09
FULL, FULL_TALL, FULL_SHORT = (TEXTWIDTH, 2.0), (TEXTWIDTH, 2.35), (TEXTWIDTH, 1.7)
HALF, HALF_TALL = (2.70, 2.05), (2.70, 2.45)

def use_course_style():
    mpl.rcParams.update({'figure.dpi': 130, 'font.size': 9, 'axes.titlecolor': BLUE,
                         'axes.edgecolor': MUTED, 'axes.linewidth': .7,
                         'axes.spines.top': False, 'axes.spines.right': False,
                         'xtick.color': MUTED, 'ytick.color': MUTED,
                         'legend.frameon': False, 'lines.linewidth': 1.8,
                         'axes.prop_cycle': mpl.cycler(color=CYCLE)})

def save(fig, *paths):     # the deck writes files; the notebook shows the plot
    plt.show()
    return Path(paths[0]) if paths else Path('figure.pdf')

use_course_style()
# The deck scripts derive output paths from OUT and NOTES. Nothing is written
# here, so both point at the working directory and the notebook runs anywhere.
OUT = Path('.')
NOTES = Path('.')

In [ ]:
from __future__ import annotations

import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle





SEED = 265
rng = np.random.default_rng(SEED)


def standardized(x: np.ndarray) -> tuple[np.ndarray, float, float]:
    center, scale = x.mean(), x.std()
    return (x - center) / scale, center, scale


def polynomial_fit(
    x: np.ndarray, y: np.ndarray, grid: np.ndarray, degree: int
) -> np.ndarray:
    z, center, scale = standardized(x)
    coef = np.polyfit(z, y, degree)
    return np.polyval(coef, (grid - center) / scale)


def kernel_fit(
    x: np.ndarray, y: np.ndarray, grid: np.ndarray, bandwidth: float
) -> np.ndarray:
    weights = np.exp(-0.5 * ((grid[:, None] - x[None, :]) / bandwidth) ** 2)
    return (weights @ y) / np.maximum(weights.sum(axis=1), 1e-12)


def wage_points(ax: plt.Axes, *, alpha: float = 0.23) -> None:
    ax.scatter(
        x_wage,
        y_wage,
        s=7,
        color=INK,
        alpha=alpha,
        edgecolors="none",
        rasterized=True,
        label="observed workers",
    )


wage = pd.read_csv(Path("Wage.csv"))
x_wage = wage["age"].to_numpy(float)
y_wage = wage["wage"].to_numpy(float)
age_grid = np.linspace(x_wage.min(), x_wage.max(), 300)


# Raw observations appear before any fitted model is introduced.
fig, ax = plt.subplots(figsize=FULL_TALL)
wage_points(ax, alpha=0.27)
ax.set(xlabel="Age (years)", ylabel="Annual wage (thousands of dollars)")
ax.set_xlim(17, 81)
save(fig, OUT / "wage-data.pdf", OUT / "wage-data.svg")

In [ ]:
# The first fitted comparison is shown only after each family has been defined.
fig, axes = plt.subplots(1, 3, figsize=FULL_TALL, sharex=True, sharey=True)
fits = [
    (polynomial_fit(x_wage, y_wage, age_grid, 1), "Linear: degree 1", BLUE),
    (polynomial_fit(x_wage, y_wage, age_grid, 4), "Polynomial: degree 4", RED),
    (kernel_fit(x_wage, y_wage, age_grid, 5.0), "Kernel: $h=5$ years", GREEN),
]
for ax, (fit, title, color) in zip(axes, fits):
    wage_points(ax, alpha=0.23)
    ax.plot(age_grid, fit, color=color, linewidth=2.4)
    ax.set(title=title, xlabel="Age")
    ax.set_xlim(17, 81)
axes[0].set_ylabel("Wage ($1000s)")
fig.tight_layout(w_pad=0.55)
save(fig, OUT / "model-families.pdf", OUT / "model-families.svg")

In [ ]:
# Fixed polynomial features on a comparable input domain.
z = np.linspace(-1, 1, 250)
fig, axes = plt.subplots(1, 4, figsize=FULL_TALL, sharex=True)
for degree, ax, color in zip(range(4), axes, [MUTED, BLUE, RED, GREEN]):
    ax.axhline(0, color=RULE, linewidth=0.8)
    ax.plot(z, z**degree, color=color, linewidth=2.3)
    ax.set_title(rf"$\phi_{degree}(z)=z^{degree}$")
    ax.set_xlabel("scaled input $z$")
    ax.set_xticks([-1, 0, 1])
    ax.set_ylim(-1.1, 1.1)
axes[0].set_ylabel("feature value")
fig.tight_layout(w_pad=0.55)
save(fig, OUT / "polynomial-basis.pdf", OUT / "polynomial-basis.svg")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=FULL_TALL, sharex=True, sharey=True)
for ax, degree, color in zip(axes, [1, 4, 10], [BLUE, RED, GREEN]):
    wage_points(ax, alpha=0.21)
    ax.plot(
        age_grid,
        polynomial_fit(x_wage, y_wage, age_grid, degree),
        color=color,
        linewidth=2.4,
    )
    ax.set(title=f"Degree {degree}", xlabel="Age")
    ax.set_xlim(17, 81)
    ax.set_ylim(0, 330)
axes[0].set_ylabel("Wage ($1000s)")
fig.tight_layout(w_pad=0.55)
save(fig, OUT / "polynomial-fits.pdf", OUT / "polynomial-fits.svg")

In [ ]:
# Four common nonnegative, symmetric kernel shapes.
u = np.linspace(-3.2, 3.2, 500)
kernels = [
    ("Uniform", 0.5 * (np.abs(u) <= 1), BLUE),
    ("Triangular", np.maximum(1 - np.abs(u), 0), RED),
    ("Epanechnikov", 0.75 * np.maximum(1 - u**2, 0), GREEN),
    ("Gaussian", np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi), BLUE_BRIGHT),
]
fig, axes = plt.subplots(1, 4, figsize=FULL_TALL, sharex=True, sharey=True)
for ax, (name, values, color) in zip(axes, kernels):
    ax.axhline(0, color=RULE, linewidth=0.8)
    ax.axvline(0, color=RULE, linewidth=0.8)
    ax.plot(u, values, color=color, linewidth=2.3)
    ax.fill_between(u, values, color=color, alpha=0.10)
    ax.set(title=name, xlabel="$u=(x-x_0)/h$", xlim=(-3, 3), ylim=(-0.04, 1.05))
    ax.set_xticks([-2, 0, 2])
axes[0].set_ylabel("weight $K(u)$")
fig.tight_layout(w_pad=0.55)
save(fig, OUT / "common-kernels.pdf", OUT / "common-kernels.svg")

In [ ]:
# Local-constant fitting at one query point, using binned means only for
# legibility; the estimate itself uses every observed row.
x0, h = 40.0, 5.0
weights = np.exp(-0.5 * ((x_wage - x0) / h) ** 2)
local_mean = float(weights @ y_wage / weights.sum())
bins = np.arange(18, 82, 2)
centers = (bins[:-1] + bins[1:]) / 2
ids = np.digitize(x_wage, bins) - 1
bin_means = np.array(
    [y_wage[ids == j].mean() if np.any(ids == j) else np.nan for j in range(len(centers))]
)

fig, axes = plt.subplots(1, 2, figsize=FULL_TALL)
axes[0].plot(age_grid, np.exp(-0.5 * ((age_grid - x0) / h) ** 2), color=MORANDI_EDGE[MORANDI_BLUE])
axes[0].axvline(x0, color=MORANDI_EDGE[MORANDI_CLAY], linestyle="--", linewidth=1.4)
axes[0].set(xlabel="Age", ylabel="relative weight", title="Weights around $x_0=40$")
axes[0].annotate("query", (x0, 1), xytext=(47, 0.84), arrowprops={"arrowstyle": "->", "color": RED}, color=RED)
axes[1].scatter(centers, bin_means, s=24, color=INK, alpha=0.75, label="two-year bin mean")
axes[1].axhline(local_mean, color=MORANDI_EDGE[MORANDI_CLAY], linewidth=2.2, label=rf"weighted mean = {local_mean:.1f}")
axes[1].axvspan(x0 - h, x0 + h, color=MORANDI_BLUE, alpha=0.09, label=r"$x_0\pm h$")
axes[1].set(xlabel="Age", ylabel="Wage ($1000s)", title="Fit one constant near the query")
axes[1].legend(loc="lower center")
fig.tight_layout(w_pad=1.0)
save(fig, OUT / "local-constant.pdf", OUT / "local-constant.svg")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=FULL_TALL, sharex=True, sharey=True)
for ax, bandwidth in zip(axes, [1.5, 5.0, 15.0]):
    wage_points(ax, alpha=0.21)
    ax.plot(age_grid, kernel_fit(x_wage, y_wage, age_grid, bandwidth), color=MORANDI_EDGE[MORANDI_CLAY], linewidth=2.4)
    ax.set(title=f"$h={bandwidth:g}$ years", xlabel="Age")
    ax.set_xlim(17, 81)
axes[0].set_ylabel("Wage ($1000s)")
fig.tight_layout(w_pad=0.55)
save(fig, OUT / "bandwidth-sequence.pdf", OUT / "bandwidth-sequence.svg")

In [ ]:
# Quantify why a local neighborhood becomes nonlocal in many dimensions.
dims = np.arange(1, 21)
side_for_ten_percent = 0.1 ** (1 / dims)
fig, axes = plt.subplots(1, 2, figsize=FULL_TALL)
axes[0].plot(dims, 0.5**dims, color=MORANDI_EDGE[MORANDI_BLUE], marker="o", markersize=3.5)
axes[0].set_yscale("log")
axes[0].set(xlabel="Dimension $d$", ylabel="fraction of volume", title="A side-$0.5$ neighborhood", xticks=[1, 5, 10, 15, 20])
axes[0].axhline(0.1, color=MORANDI_EDGE[MORANDI_CLAY], linestyle="--", linewidth=1.1)
axes[1].plot(dims, side_for_ten_percent, color=MORANDI_EDGE[MORANDI_SAGE], marker="o", markersize=3.5)
axes[1].set(xlabel="Dimension $d$", ylabel="required side length $h$", title="To include 10% of the cube", xticks=[1, 5, 10, 15, 20], ylim=(0, 1.03))
for d in [1, 5, 10, 20]:
    axes[1].annotate(f"{side_for_ten_percent[d-1]:.2f}", (d, side_for_ten_percent[d-1]), xytext=(0, 7), textcoords="offset points", ha="center", color=GREEN)
fig.tight_layout(w_pad=1.0)
save(fig, OUT / "curse-volume.pdf", OUT / "curse-volume.svg")

In [ ]:
# Empirical nearest-neighbor distance distribution in a unit cube.
fig, ax = plt.subplots(figsize=FULL_TALL)
for d, color in zip([1, 2, 5, 10], [BLUE, RED, GREEN, MUTED]):
    train = rng.uniform(size=(1000, d))
    query = rng.uniform(size=(250, d))
    distances = np.sqrt(((query[:, None, :] - train[None, :, :]) ** 2).sum(axis=2)).min(axis=1)
    sorted_d = np.sort(distances)
    ax.plot(sorted_d, np.linspace(0, 1, len(sorted_d), endpoint=False), color=color, label=f"$d={d}$")
ax.set(xlabel="distance to nearest of 1,000 observations", ylabel="fraction of query points", title="Nearest observations become farther away")
ax.legend(loc="lower right", ncol=2)
save(fig, OUT / "nearest-distance.pdf", OUT / "nearest-distance.svg")

In [ ]:
# Three activation functions used in neural networks.  The exact GELU is
# z Phi(z), where Phi is the standard normal CDF.
activation_grid = np.linspace(-4, 4, 600)
relu = np.maximum(activation_grid, 0)
gelu = 0.5 * activation_grid * (
    1 + np.array([math.erf(value / np.sqrt(2)) for value in activation_grid])
)
sigmoid = 1 / (1 + np.exp(-activation_grid))
fig, axes = plt.subplots(1, 3, figsize=FULL_SHORT, sharex=True)
for ax, values, title, formula, color in zip(
    axes,
    [relu, gelu, sigmoid],
    ["ReLU", "GELU", "sigmoid"],
    [
        r"$\sigma(z)=\max\{0,z\}$",
        r"$\sigma(z)=z\Phi(z)$",
        r"$\sigma(z)=(1+e^{-z})^{-1}$",
    ],
    [BLUE, GREEN, RED],
):
    ax.axhline(0, color=RULE, linewidth=0.8)
    ax.axvline(0, color=RULE, linewidth=0.8)
    ax.plot(activation_grid, values, color=color, linewidth=2.3)
    ax.set(title=f"{title}\n{formula}", xlabel="input $z$", xlim=(-4, 4))
    ax.set_xticks([-4, 0, 4])
axes[0].set_ylabel(r"output $\sigma(z)$")
fig.tight_layout(w_pad=0.55)
save(fig, OUT / "activation-functions.pdf", OUT / "activation-functions.svg")

In [ ]:
# Functions represented by a one-hidden-layer fully connected network.  These
# are model-class members, not fitted Wage curves.
z_grid = np.linspace(-2.5, 2.5, 400)
fig, ax = plt.subplots(figsize=FULL_TALL)
for j, color in enumerate([BLUE, RED, GREEN]):
    local_rng = np.random.default_rng(SEED + j)
    width = 5
    a = local_rng.normal(size=width)
    w = local_rng.normal(size=width)
    b = local_rng.normal(size=width)
    values = (np.maximum(0, z_grid[:, None] * w + b) @ a) / 2.5
    ax.plot(z_grid, values, color=color)
ax.set(xlabel="input $x$", ylabel="$f(x)$", title="Members of a width-5 ReLU network class")
save(fig, OUT / "network-functions.pdf", OUT / "network-functions.svg")

In [ ]:
# Refit on random subsets without replacement to show sample sensitivity using
# only the sampling idea already introduced through data splitting.
fig, axes = plt.subplots(1, 2, figsize=FULL_TALL, sharex=True, sharey=True)
for ax, degree in zip(axes, [1, 10]):
    for _ in range(24):
        idx = rng.choice(len(wage), size=600, replace=False)
        fit = polynomial_fit(x_wage[idx], y_wage[idx], age_grid, degree)
        ax.plot(age_grid, fit, color=MORANDI_EDGE[MORANDI_BLUE], alpha=0.20, linewidth=0.8)
    full_fit = polynomial_fit(x_wage, y_wage, age_grid, degree)
    ax.plot(age_grid, full_fit, color=MORANDI_EDGE[MORANDI_CLAY], linewidth=2.2)
    ax.set(title=f"Degree {degree}", xlabel="Age", xlim=(17, 81), ylim=(0, 330))
axes[0].set_ylabel("Fitted wage")
fig.tight_layout(w_pad=0.75)
save(fig, OUT / "repeated-fits.pdf", OUT / "repeated-fits.svg")

In [ ]:
# A real held-out comparison on Auto: polynomial horsepower models for MPG.
auto = pd.read_csv(Path("Auto.csv"))
auto["horsepower"] = pd.to_numeric(auto["horsepower"], errors="coerce")
auto = auto.dropna(subset=["horsepower", "mpg"])
x_auto = auto["horsepower"].to_numpy(float)
y_auto = auto["mpg"].to_numpy(float)
order = rng.permutation(len(auto))
cut = int(0.70 * len(auto))
train, test = order[:cut], order[cut:]
degrees = np.arange(1, 11)
train_mse, test_mse = [], []
for degree in degrees:
    center, scale = x_auto[train].mean(), x_auto[train].std()
    coef = np.polyfit((x_auto[train] - center) / scale, y_auto[train], degree)
    train_pred = np.polyval(coef, (x_auto[train] - center) / scale)
    test_pred = np.polyval(coef, (x_auto[test] - center) / scale)
    train_mse.append(np.mean((y_auto[train] - train_pred) ** 2))
    test_mse.append(np.mean((y_auto[test] - test_pred) ** 2))

fig, ax = plt.subplots(figsize=FULL_TALL)
ax.plot(degrees, train_mse, color=MORANDI_EDGE[MORANDI_BLUE], marker="o", label="training")
ax.plot(degrees, test_mse, color=MORANDI_EDGE[MORANDI_CLAY], marker="o", label="held-out")
ax.set(xlabel="Polynomial degree", ylabel="Mean squared error", title="MPG from horsepower", xticks=degrees)
ax.legend()
save(fig, OUT / "complexity-errors.pdf", OUT / "complexity-errors.svg")

In [ ]:
# In-range and out-of-range behavior for two fitted rules.
grid_ext = np.linspace(5, 95, 380)
poly4 = polynomial_fit(x_wage, y_wage, grid_ext, 4)
kernel5 = kernel_fit(x_wage, y_wage, grid_ext, 5.0)
fig, ax = plt.subplots(figsize=FULL_TALL)
wage_points(ax, alpha=0.18)
ax.axvspan(18, 80, color=RULE, alpha=0.22, label="observed age range")
ax.plot(grid_ext, poly4, color=MORANDI_EDGE[MORANDI_CLAY], label="degree 4 polynomial")
ax.plot(grid_ext, kernel5, color=MORANDI_EDGE[MORANDI_SAGE], label="Gaussian kernel, $h=5$")
ax.set(xlabel="Age", ylabel="Predicted wage ($1000s)", title="Extrapolation is determined by the model class", xlim=(5, 95))
ax.legend(loc="upper left")
save(fig, OUT / "extrapolation.pdf", OUT / "extrapolation.svg")

In [ ]:
print(f"Wage rows: {len(wage)}; age range: {x_wage.min():.0f}--{x_wage.max():.0f}")
print(f"Local-constant estimate at age {x0:.0f}, h={h:.0f}: {local_mean:.2f}")
print(f"Auto rows after cleaning: {len(auto)}")
print(f"Best held-out polynomial degree: {degrees[int(np.argmin(test_mse))]}")


def local_average_figure():
    """The naive local average: a hard-edged neighborhood around one query."""
    x0, h = 40.0, 5.0
    inside = np.abs(x_wage - x0) <= h
    est = y_wage[inside].mean()
    fig, axes = plt.subplots(1, 2, figsize=(FULL[0], 1.95), sharey=True)
    # Left: one query, its neighborhood, its average.
    ax = axes[0]
    ax.scatter(x_wage[~inside], y_wage[~inside], s=6, color=MUTED, alpha=.18,
               edgecolors="none", rasterized=True)
    ax.scatter(x_wage[inside], y_wage[inside], s=8, color=MORANDI_BLUE, alpha=.55,
               edgecolors="none", rasterized=True)
    ax.axvspan(x0 - h, x0 + h, color=MORANDI_BLUE, alpha=.07)
    ax.plot([x0 - h, x0 + h], [est, est], color=MORANDI_EDGE[MORANDI_CLAY], linewidth=2)
    ax.scatter([x0], [est], color=MORANDI_CLAY, s=22, zorder=5)
    ax.set(xlabel="age", ylabel="wage (\\$1000s)", ylim=(0, 320),
           title=f"one query: average of {inside.sum()} points")
    # Right: repeating it at every query gives a step function.
    # A narrower window makes the hard edge visible: at h=5 each neighbourhood
    # holds ~500 rows and the steps are far too small to see.
    h2 = 0.75
    grid = np.linspace(x_wage.min(), x_wage.max(), 400)
    curve = np.array([y_wage[np.abs(x_wage - g) <= h2].mean()
                      if np.any(np.abs(x_wage - g) <= h2) else np.nan for g in grid])
    ax = axes[1]
    ax.scatter(x_wage, y_wage, s=6, color=MUTED, alpha=.18, edgecolors="none",
               rasterized=True)
    ax.plot(grid, curve, color=MORANDI_EDGE[MORANDI_CLAY], linewidth=1.1)
    ax.set(xlabel="age", ylim=(0, 320),
           title=f"every query, $h={h2}$: steps as points enter and leave")
    fig.tight_layout(pad=0.3, w_pad=0.5)
    print(f"local average at age {x0:.0f} with h={h:.0f}: {est:.1f}, "
          f"{inside.sum()} of {len(x_wage)} points")
    return save(fig, OUT / "local-average.pdf", OUT / "local-average.svg")


def knn_figure():
    """k-nearest-neighbour regression at three values of k."""
    grid = np.linspace(x_wage.min(), x_wage.max(), 400)
    order = np.argsort(np.abs(grid[:, None] - x_wage[None, :]), axis=1)
    fig, axes = plt.subplots(1, 3, figsize=(FULL[0], 1.95), sharey=True)
    for ax, k in zip(axes, [5, 50, 500]):
        curve = y_wage[order[:, :k]].mean(axis=1)
        ax.scatter(x_wage, y_wage, s=6, color=MUTED, alpha=.18, edgecolors="none",
                   rasterized=True)
        ax.plot(grid, curve, color=MORANDI_EDGE[MORANDI_BLUE], linewidth=1.4)
        ax.set(xlabel="age", ylim=(0, 320), title=f"$k={k}$")
    axes[0].set_ylabel("wage (\\$1000s)")
    fig.tight_layout(pad=0.3, w_pad=0.5)
    return save(fig, OUT / "knn-fits.pdf", OUT / "knn-fits.svg")


def curse_knn_figure():
    """KNN against linear regression as noise predictors are added.

    Follows ISL Figure 3.20: the response depends only on the first feature, and
    the extra features are pure noise.  Linear regression barely notices them;
    KNN degrades sharply, because with a fixed n the neighbours of a query stop
    being near it.  This replaces an abstract unit-cube volume calculation.
    """
    gen = np.random.default_rng(SEED)
    n_train, n_test, reps = 50, 500, 60
    dims = [1, 2, 3, 4, 5, 10, 20]
    lin_mse, knn_mse = [], []
    for p in dims:
        lin_r, knn_r = [], []
        for _ in range(reps):
            Xtr = gen.uniform(-1, 1, (n_train, p))
            Xte = gen.uniform(-1, 1, (n_test, p))
            f = lambda X: np.sin(2.0 * X[:, 0]) + 0.3 * X[:, 0]   # depends on x1 only
            ytr = f(Xtr) + 0.2 * gen.standard_normal(n_train)
            yte = f(Xte)
            # least squares with an intercept
            A = np.column_stack([np.ones(n_train), Xtr])
            beta, *_ = np.linalg.lstsq(A, ytr, rcond=None)
            pred_lin = np.column_stack([np.ones(n_test), Xte]) @ beta
            lin_r.append(np.mean((yte - pred_lin) ** 2))
            # k-nearest neighbours, k = 5
            d = np.linalg.norm(Xte[:, None, :] - Xtr[None, :, :], axis=2)
            idx = np.argsort(d, axis=1)[:, :5]
            pred_knn = ytr[idx].mean(axis=1)
            knn_r.append(np.mean((yte - pred_knn) ** 2))
        lin_mse.append(np.mean(lin_r)); knn_mse.append(np.mean(knn_r))
    fig, ax = plt.subplots(figsize=(FULL[0], 1.95))
    # Log scale on both axes: on a linear y-axis the least-squares curve is
    # pinned to zero and the crossover at small p -- the whole point -- is
    # invisible.
    ax.plot(dims, knn_mse, "o-", color=MORANDI_EDGE[MORANDI_CLAY], markersize=3.4, label="$k$-nearest neighbours ($k=5$)")
    ax.plot(dims, lin_mse, "o-", color=MORANDI_EDGE[MORANDI_BLUE], markersize=3.4, label="linear regression")
    cross = next((d for d, a, b in zip(dims, knn_mse, lin_mse) if a > b), None)
    if cross is not None:
        ax.axvline(cross, color=MUTED, linestyle=":", linewidth=1)
        ax.annotate(f"KNN falls behind\nat $p={cross}$", xy=(cross, lin_mse[dims.index(cross)]),
                    xytext=(cross * 1.35, min(lin_mse) * 1.4), color=MUTED, fontsize=6.6,
                    arrowprops=dict(arrowstyle="->", color=MUTED, lw=.7))
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set(xlabel="number of features $p$ (only the first one matters)",
           ylabel="test MSE", title=f"$n={n_train}$ training points")
    ax.set_xticks(dims); ax.set_xticklabels([str(d) for d in dims])
    ax.legend(framealpha=0, loc="upper left")
    fig.tight_layout(pad=0.3)
    print("curse: p =", dims)
    print("  knn   ", [round(v, 3) for v in knn_mse])
    print("  linear", [round(v, 3) for v in lin_mse])
    print(f"  knn MSE grows {knn_mse[-1]/knn_mse[0]:.1f}x from p=1 to p=20; "
          f"linear {lin_mse[-1]/lin_mse[0]:.1f}x")
    return save(fig, OUT / "curse-knn.pdf", OUT / "curse-knn.svg")




# Calls preserved from the script's __main__ block.
local_average_figure()

In [ ]:
knn_figure()

In [ ]:
curse_knn_figure()